In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)
from simulation.simulator import (
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2026-01-01"
sim_start = "2026-05-06"
end = None

In [ ]:
# ============================================================
# RSI Configs
# ============================================================

RSI_DIV_CFG = {
    "length": 14,
    "pivot_lookback": 5,
    "min_rsi_delta": 2,

    # divergence filtering
    "os_level": 30,
    "ob_level": 70,
    "zone_mode": "cross",

    # panel styling
    "major_levels": [30, 70],
    "minor_levels": [20, 80],
    "show_zone_shading": True,

    # divergence labels
    "show_div_labels": True,
    "max_div_labels": 6,
    "label_font_size": 10,
    "label_xshift": 10,
    "label_yshift": 14,

    # main chart markers
    "mark_price": True,
    "price_marker_size": 18,
    "marker_offset_mult": 1.35,
}

# ============================================================
# Supertrend Configs
# ============================================================

ST1_1M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST1_5M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST1_15M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST2_1M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}

ST2_5M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}

ST2_15M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}


# ============================================================
# Supertrend Tags
# ============================================================

ST1_TAG = "st1_21_1"
ST2_TAG = "st2_14_2"

In [ ]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"

### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"

### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"


In [ ]:
# ============================================================
# 1m indicators
# ============================================================

ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_1M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_1M_CFG),
]


# ============================================================
# 5m indicators
# ============================================================

ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_5M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_5M_CFG),
]


# ============================================================
# 15m indicators
# ============================================================

ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_15M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_15M_CFG),
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
    },
    base_label="1m",
)


In [ ]:
# ============================================================
# Helper to build Supertrend column names
# ============================================================

def st_cols(tf: str, tag: str) -> dict:
    root = f"{tf}__{tag}"
    return {
        "ST": f"{root}__ST",
        "DIR": f"{root}__DIR",
        "BUY": f"{root}__BUY",
        "SELL": f"{root}__SELL",
        "BUY_LINE": f"{root}__ST_BUY_LINE",
        "SELL_LINE": f"{root}__ST_SELL_LINE",
    }


# ============================================================
# ST1 aliases: length 21, multiplier 1.0
# ============================================================

ST1_1M = st_cols("1m", ST1_TAG)
ST1_5M = st_cols("5m", ST1_TAG)
ST1_15M = st_cols("15m", ST1_TAG)


# ============================================================
# ST2 aliases: length 14, multiplier 2.0
# ============================================================

ST2_1M = st_cols("1m", ST2_TAG)
ST2_5M = st_cols("5m", ST2_TAG)
ST2_15M = st_cols("15m", ST2_TAG)


In [ ]:
[c for c in merged.columns if "supertrend" in c.lower()]

In [ ]:
N_CONFIRM = 1               # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 100
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 10 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


#####################################################################################


LONG_EMA_FILTERS = [
    # EMA50_1m,
    # EMA100_1m,
    # EMA150_1m,
    # EMA200_1m,
    # EMA100_5m,
    # EMA200_5m,
    # EMA100_15m,
    EMA200_15m
]


macd_rules = ALL(
    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, 10)
    ),
    
    # Rule(
    #     "5m MACD above threshold",
    #     lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
    # ),
    
    # Rule(
    #     "15m MACD above threshold",
    #     lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
    # ),

    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_1M, 50)
    ),
    
    Rule(
         "5m MACD - Signal above minimum",
         lambda c: c.gt(MACD_HIST_5M, 20)
     ),

    Rule(
        "15m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_15M, 20)
    ),
    
)

supertrend_long_filter_strict = ALL(
    # Rule("ST1 1m bullish", lambda c: c.gt(ST1_1M["DIR"], 0)),
    # Rule("ST1 5m bullish", lambda c: c.gt(ST1_1M["DIR"], 0)),
    Rule("ST1 15m bullish", lambda c: c.gt(ST1_15M["DIR"], 0)),

    # Rule("ST2 1m bullish", lambda c: c.gt(ST2_1M["DIR"], 0)),
    # Rule("ST2 5m bullish", lambda c: c.gt(ST2_1M["DIR"], 0)),
    Rule("ST2 15m bullish", lambda c: c.gt(ST2_15M["DIR"], 0)),
)

open_rules_long = ALL(
    supertrend_long_filter_strict,
    macd_rules,
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)

),
    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),
)

close_rules_long = ALL(
    # Rule(
    #     "ST1 1m bearish",
    #     lambda c: c.lt(ST1_15M["DIR"], 0)
    # ),
    Rule(
        "ST2 5m bearish",
        lambda c: c.lt(ST2_5M["DIR"], 0)
    ),
)



###################### short rules #########################


In [ ]:
# ============================================================
# FAST / REDUCED HOLISTIC TP-SL OPTIMIZER
# Target: ~1,232 simulations
# Includes:
#   - fixed TP
#   - RR dynamic TP
#   - partial exits
#   - breakeven stop moves
#   - runner exits
#   - allow_rule_close True/False
#   - precomputed open/close signals for speed
# ============================================================

import time
import itertools
from pathlib import Path
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from simulation.rules import Rule, ALL
from simulation.simulator import (
    Strategy,
    SimConfig,
    run_simulation,
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

# ============================================================
# Main settings
# ============================================================

INITIAL_CASH = 10_000
RESULTS_PATH = Path("optimization_results_exit_strategy_fast_2464.csv")
CHECKPOINT_EVERY = 25
MIN_TRADES_FOR_RANKING = 8

SIM_START = "2026-01-01"
SIM_END = "2026-05-18"

FEE_BPS = 10
SLIPPAGE_BPS = 1
MAX_OPEN_TRADES = 1

# ============================================================
# Stop-loss options
# 0.10 = 0.10% from entry
# ============================================================

STOP_LOSS_PCTS = (
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.75,
    1.0,
)

ALLOW_RULE_CLOSE_VALUES = (
    False,
    True,
)

# Keep sizing fixed for this pass.
# This keeps the grid focused on exit quality, not position sizing.
SIZING_CONFIG = {
    "name": "cash",
    "config": PositionSizingConfig(mode="cash"),
}

# ============================================================
# Precompute fixed strategy signals once
# Update these two lines if your entry/close strategy changes.
# ============================================================

merged["__fixed_open_signal"] = (
    (merged[ST1_15M["DIR"]].to_numpy(dtype=float) > 0)
    & (merged[ST2_15M["DIR"]].to_numpy(dtype=float) > 0)
)

merged["__fixed_close_signal"] = (
    merged[ST2_5M["DIR"]].to_numpy(dtype=float) < 0
)

fast_strategy = Strategy(
    open_rules_long=ALL(
        Rule("Precomputed fixed open signal", lambda c: c.flag("__fixed_open_signal"))
    ),
    close_rules_long=ALL(
        Rule("Precomputed fixed close signal", lambda c: c.flag("__fixed_close_signal"))
    ),
)

# ============================================================
# Exit template generation
# ============================================================

def _pct_name(x: float) -> str:
    return str(float(x)).replace(".", "p")


def _rr_name(x: float) -> str:
    return str(float(x)).replace(".", "p") + "R"


def _tp(label, mode, value, close_pct=100.0, move_stop_mode="none", move_stop_value=0.0):
    return TakeProfitConfig(
        label=label,
        mode=mode,
        value=float(value),
        close_pct=float(close_pct),
        move_stop_mode=move_stop_mode,
        move_stop_value=float(move_stop_value),
    )


def build_exit_plans_fast():
    """
    Builds 154 diverse templates:
      Original 77:
        - single fixed
        - single RR
        - two-stage fixed
        - two-stage RR
        - three-stage fixed
        - runner

      New 77:
        - scalp lock exits
        - mixed fixed + RR exits
        - front-loaded exits
        - back-loaded exits
        - wider runners
        - micro runners
    """
    plans = []

    def add(name, category, tps):
        plans.append({
            "name": name,
            "category": category,
            "take_profits": tuple(tps),
        })

    # ========================================================
    # A) Single fixed TP - 8
    # ========================================================
    for tp in (0.20, 0.30, 0.40, 0.50, 0.75, 1.00, 1.50, 2.00):
        add(
            name=f"single_fixed_tp_{_pct_name(tp)}",
            category="single_fixed",
            tps=[
                _tp("TP1", "entry_pct", tp, close_pct=100.0),
            ],
        )

    # ========================================================
    # B) Single RR TP - 6
    # ========================================================
    for rr in (0.75, 1.00, 1.25, 1.50, 2.00, 3.00):
        add(
            name=f"single_rr_{_rr_name(rr)}",
            category="single_rr",
            tps=[
                _tp(f"TP1_{_rr_name(rr)}", "rr", rr, close_pct=100.0),
            ],
        )

    # ========================================================
    # C) Two-stage fixed - 24
    # ========================================================
    for tp1 in (0.30, 0.50, 0.75):
        for tp2 in (1.00, 1.50):
            for close1 in (50.0, 67.0):
                for move_mode in ("none", "breakeven"):
                    add(
                        name=(
                            f"two_fixed_tp1_{_pct_name(tp1)}_c{int(close1)}_"
                            f"{move_mode}_tp2_{_pct_name(tp2)}"
                        ),
                        category="two_fixed",
                        tps=[
                            _tp(
                                "TP1",
                                "entry_pct",
                                tp1,
                                close_pct=close1,
                                move_stop_mode=move_mode,
                            ),
                            _tp(
                                "TP2",
                                "entry_pct",
                                tp2,
                                close_pct=100.0,
                            ),
                        ],
                    )

    # ========================================================
    # D) Two-stage RR - 16
    # ========================================================
    for rr1 in (0.75, 1.00):
        for rr2 in (1.50, 2.00):
            for close1 in (50.0, 67.0):
                for move_mode in ("none", "breakeven"):
                    add(
                        name=(
                            f"two_rr_tp1_{_rr_name(rr1)}_c{int(close1)}_"
                            f"{move_mode}_tp2_{_rr_name(rr2)}"
                        ),
                        category="two_rr",
                        tps=[
                            _tp(
                                f"TP1_{_rr_name(rr1)}",
                                "rr",
                                rr1,
                                close_pct=close1,
                                move_stop_mode=move_mode,
                            ),
                            _tp(
                                f"TP2_{_rr_name(rr2)}",
                                "rr",
                                rr2,
                                close_pct=100.0,
                            ),
                        ],
                    )

    # ========================================================
    # E) Three-stage fixed - 18
    # ========================================================
    fixed_ladders = (
        (0.30, 0.75, 1.50),
        (0.50, 1.00, 2.00),
        (0.75, 1.50, 3.00),
    )

    close_patterns = (
        (25.0, 50.0),
        (33.0, 50.0),
        (50.0, 50.0),
    )

    stop_moves = (
        ("breakeven", 0.0, "be"),
        ("entry_pct", 0.05, "plus0p05"),
    )

    for tp1, tp2, tp3 in fixed_ladders:
        for close1, close2 in close_patterns:
            for move_mode, move_value, move_name in stop_moves:
                add(
                    name=(
                        f"three_fixed_{_pct_name(tp1)}_{_pct_name(tp2)}_{_pct_name(tp3)}_"
                        f"c{int(close1)}_{int(close2)}_{move_name}"
                    ),
                    category="three_fixed",
                    tps=[
                        _tp(
                            "TP1",
                            "entry_pct",
                            tp1,
                            close_pct=close1,
                            move_stop_mode=move_mode,
                            move_stop_value=move_value,
                        ),
                        _tp(
                            "TP2",
                            "entry_pct",
                            tp2,
                            close_pct=close2,
                        ),
                        _tp(
                            "TP3",
                            "entry_pct",
                            tp3,
                            close_pct=100.0,
                        ),
                    ],
                )

    # ========================================================
    # F) Runner - 5
    # ========================================================
    for tp1, close1, final_tp in (
        (0.30, 33.0, 1.50),
        (0.40, 33.0, 2.00),
        (0.50, 33.0, 3.00),
        (0.75, 50.0, 3.00),
        (1.00, 50.0, 4.00),
    ):
        add(
            name=f"runner_tp1_{_pct_name(tp1)}_c{int(close1)}_be_final_{_pct_name(final_tp)}",
            category="runner",
            tps=[
                _tp(
                    "TP1",
                    "entry_pct",
                    tp1,
                    close_pct=close1,
                    move_stop_mode="breakeven",
                ),
                _tp(
                    "RUNNER",
                    "entry_pct",
                    final_tp,
                    close_pct=100.0,
                ),
            ],
        )

    # ========================================================
    # G) New: scalp lock fixed exits - 16
    # Small TP1, quickly protect trade, modest final target.
    # ========================================================
    for tp1 in (0.20, 0.30, 0.40, 0.50):
        for tp2 in (0.75, 1.00):
            for close1 in (50.0, 67.0):
                add(
                    name=(
                        f"scalp_lock_tp1_{_pct_name(tp1)}_c{int(close1)}_"
                        f"plus0p03_tp2_{_pct_name(tp2)}"
                    ),
                    category="scalp_lock",
                    tps=[
                        _tp(
                            "TP1",
                            "entry_pct",
                            tp1,
                            close_pct=close1,
                            move_stop_mode="entry_pct",
                            move_stop_value=0.03,
                        ),
                        _tp(
                            "TP2",
                            "entry_pct",
                            tp2,
                            close_pct=100.0,
                        ),
                    ],
                )

    # ========================================================
    # H) New: mixed fixed + RR exits - 24
    # TP1 fixed %, final TP based on R.
    # Good when first target should be price-based but runner adapts to SL.
    # ========================================================
    for tp1 in (0.30, 0.50, 0.75):
        for rr2 in (1.50, 2.00):
            for close1 in (33.0, 50.0):
                for move_mode, move_value, move_name in (
                    ("breakeven", 0.0, "be"),
                    ("entry_pct", 0.05, "plus0p05"),
                ):
                    add(
                        name=(
                            f"mixed_fixed_rr_tp1_{_pct_name(tp1)}_c{int(close1)}_"
                            f"{move_name}_tp2_{_rr_name(rr2)}"
                        ),
                        category="mixed_fixed_rr",
                        tps=[
                            _tp(
                                "TP1",
                                "entry_pct",
                                tp1,
                                close_pct=close1,
                                move_stop_mode=move_mode,
                                move_stop_value=move_value,
                            ),
                            _tp(
                                f"TP2_{_rr_name(rr2)}",
                                "rr",
                                rr2,
                                close_pct=100.0,
                            ),
                        ],
                    )

    # ========================================================
    # I) New: front-loaded fixed exits - 12
    # Take most of the position out early.
    # Good for strategies where follow-through is unreliable.
    # ========================================================
    for tp1 in (0.25, 0.40, 0.50):
        for tp2 in (1.00, 1.50):
            for close1 in (75.0, 80.0):
                add(
                    name=(
                        f"front_loaded_tp1_{_pct_name(tp1)}_c{int(close1)}_"
                        f"be_tp2_{_pct_name(tp2)}"
                    ),
                    category="front_loaded",
                    tps=[
                        _tp(
                            "TP1",
                            "entry_pct",
                            tp1,
                            close_pct=close1,
                            move_stop_mode="breakeven",
                        ),
                        _tp(
                            "TP2",
                            "entry_pct",
                            tp2,
                            close_pct=100.0,
                        ),
                    ],
                )

    # ========================================================
    # J) New: back-loaded three-stage exits - 12
    # Take small amount early, leave more size for later expansion.
    # ========================================================
    back_ladders = (
        (0.30, 1.00, 2.00),
        (0.50, 1.50, 3.00),
        (0.75, 2.00, 4.00),
    )

    back_patterns = (
        (20.0, 40.0),
        (25.0, 25.0),
    )

    for tp1, tp2, tp3 in back_ladders:
        for close1, close2 in back_patterns:
            for move_mode, move_value, move_name in (
                ("breakeven", 0.0, "be"),
                ("entry_pct", 0.05, "plus0p05"),
            ):
                add(
                    name=(
                        f"back_loaded_{_pct_name(tp1)}_{_pct_name(tp2)}_{_pct_name(tp3)}_"
                        f"c{int(close1)}_{int(close2)}_{move_name}"
                    ),
                    category="back_loaded",
                    tps=[
                        _tp(
                            "TP1",
                            "entry_pct",
                            tp1,
                            close_pct=close1,
                            move_stop_mode=move_mode,
                            move_stop_value=move_value,
                        ),
                        _tp(
                            "TP2",
                            "entry_pct",
                            tp2,
                            close_pct=close2,
                        ),
                        _tp(
                            "TP3",
                            "entry_pct",
                            tp3,
                            close_pct=100.0,
                        ),
                    ],
                )

    # ========================================================
    # K) New: wide runner exits - 8
    # Protect after TP1 and aim for larger trend legs.
    # ========================================================
    for tp1, close1, final_tp in (
        (0.40, 25.0, 3.00),
        (0.50, 25.0, 4.00),
        (0.75, 33.0, 4.00),
        (1.00, 33.0, 5.00),
    ):
        for move_mode, move_value, move_name in (
            ("breakeven", 0.0, "be"),
            ("entry_pct", 0.05, "plus0p05"),
        ):
            add(
                name=(
                    f"wide_runner_tp1_{_pct_name(tp1)}_c{int(close1)}_"
                    f"{move_name}_final_{_pct_name(final_tp)}"
                ),
                category="wide_runner",
                tps=[
                    _tp(
                        "TP1",
                        "entry_pct",
                        tp1,
                        close_pct=close1,
                        move_stop_mode=move_mode,
                        move_stop_value=move_value,
                    ),
                    _tp(
                        "RUNNER",
                        "entry_pct",
                        final_tp,
                        close_pct=100.0,
                    ),
                ],
            )

    # ========================================================
    # L) New: micro runner exits - 5
    # Very small TP1, then hold runner.
    # Useful for reducing risk quickly.
    # ========================================================
    for tp1, final_tp in (
        (0.20, 1.00),
        (0.25, 1.25),
        (0.30, 1.50),
        (0.40, 2.00),
        (0.50, 2.50),
    ):
        add(
            name=f"micro_runner_tp1_{_pct_name(tp1)}_c25_be_final_{_pct_name(final_tp)}",
            category="micro_runner",
            tps=[
                _tp(
                    "TP1",
                    "entry_pct",
                    tp1,
                    close_pct=25.0,
                    move_stop_mode="breakeven",
                ),
                _tp(
                    "RUNNER",
                    "entry_pct",
                    final_tp,
                    close_pct=100.0,
                ),
            ],
        )

    assert len(plans) == 154, f"Expected 154 exit plans, got {len(plans)}"
    assert len({p["name"] for p in plans}) == len(plans), "Duplicate exit plan names found"

    return plans

EXIT_PLANS = build_exit_plans_fast()

print(f"Exit plans: {len(EXIT_PLANS):,}")

# ============================================================
# Helpers
# ============================================================

def _to_dataframe(obj):
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()

        rows = []
        for x in obj:
            if isinstance(x, dict):
                rows.append(x)
            elif is_dataclass(x):
                rows.append(asdict(x))
            elif hasattr(x, "__dict__"):
                rows.append(vars(x))
            else:
                rows.append(x)

        return pd.DataFrame(rows)

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    return pd.DataFrame(obj)


def make_exit_cfg(stop_loss_pct: float, exit_plan: dict, allow_rule_close: bool):
    return TradeExitConfig(
        enabled=True,
        stop_loss=StopLossConfig(
            mode="entry_pct",
            value=float(stop_loss_pct),
        ),
        sizing=SIZING_CONFIG["config"],
        take_profits=exit_plan["take_profits"],
        intrabar_priority="stop_first",
        allow_rule_close=bool(allow_rule_close),
    )


def make_sim_config(exit_cfg: TradeExitConfig):
    kwargs = dict(
        initial_cash=INITIAL_CASH,
        max_open_trades=MAX_OPEN_TRADES,
        fee_bps=FEE_BPS,
        slippage_bps=SLIPPAGE_BPS,
        sim_start=SIM_START,
        sim_end=SIM_END,
        sim_tz=tz,
        exit=exit_cfg,
    )

    try:
        return SimConfig(
            **kwargs,
            log_level="WARNING",
            progress=False,
            progress_bar=False,
        )
    except TypeError:
        return SimConfig(**kwargs)


def summarize_exit_result(res, params: dict) -> dict:
    trades = _to_dataframe(getattr(res, "trades", None))
    events = _to_dataframe(getattr(res, "events", None))

    if trades.empty or "pnl" not in trades.columns:
        closed = pd.DataFrame()
    else:
        closed = trades[trades["pnl"].notna()].copy()

    num_trades = len(closed)

    if num_trades:
        pnl = float(closed["pnl"].sum())
        wins = closed[closed["pnl"] > 0]
        losses = closed[closed["pnl"] < 0]

        win_rate = float(len(wins) / num_trades * 100)
        loss_rate = float(len(losses) / num_trades * 100)

        avg_pnl = float(closed["pnl"].mean())
        median_pnl = float(closed["pnl"].median())
        avg_winner = float(wins["pnl"].mean()) if len(wins) else np.nan
        avg_loser = float(losses["pnl"].mean()) if len(losses) else np.nan
        best_trade = float(closed["pnl"].max())
        worst_trade = float(closed["pnl"].min())

        avg_bars = float(closed["bars_held"].mean()) if "bars_held" in closed.columns else np.nan
        median_bars = float(closed["bars_held"].median()) if "bars_held" in closed.columns else np.nan
    else:
        pnl = 0.0
        win_rate = 0.0
        loss_rate = 0.0
        avg_pnl = 0.0
        median_pnl = 0.0
        avg_winner = np.nan
        avg_loser = np.nan
        best_trade = np.nan
        worst_trade = np.nan
        avg_bars = np.nan
        median_bars = np.nan

    final_equity = INITIAL_CASH + pnl

    stats = getattr(res, "stats", {})
    if isinstance(stats, pd.Series):
        stats = stats.to_dict()

    if isinstance(stats, dict):
        final_equity = float(stats.get("final_cash", final_equity))
        max_dd = float(stats.get("max_drawdown_pct", np.nan))
        profit_factor = float(stats.get("profit_factor", np.nan))
        total_return_pct = float(stats.get("total_return_pct", (final_equity / INITIAL_CASH - 1) * 100))
    else:
        max_dd = np.nan
        profit_factor = np.nan
        total_return_pct = (final_equity / INITIAL_CASH - 1) * 100

    close_reason_counts = {}
    if not closed.empty and "close_reason" in closed.columns:
        close_reason_counts = closed["close_reason"].value_counts(dropna=False).to_dict()

    partial_count = 0
    move_stop_count = 0
    if not events.empty and "event" in events.columns:
        partial_count = int((events["event"] == "PARTIAL_CLOSE").sum())
        move_stop_count = int((events["event"] == "MOVE_STOP").sum())

    if np.isfinite(max_dd):
        score = pnl - (max_dd * 25.0)
    else:
        score = pnl

    if np.isfinite(profit_factor):
        score += min(profit_factor, 5.0) * 25.0

    if num_trades < MIN_TRADES_FOR_RANKING:
        score -= 500.0

    return {
        **params,

        "num_trades": num_trades,
        "win_rate": win_rate,
        "loss_rate": loss_rate,
        "pnl": pnl,
        "final_equity": final_equity,
        "return_pct": total_return_pct,

        "profit_factor": profit_factor,
        "max_drawdown_pct": max_dd,

        "avg_pnl": avg_pnl,
        "median_pnl": median_pnl,
        "avg_winner": avg_winner,
        "avg_loser": avg_loser,
        "best_trade": best_trade,
        "worst_trade": worst_trade,

        "avg_bars_held": avg_bars,
        "median_bars_held": median_bars,

        "partial_exit_events": partial_count,
        "move_stop_events": move_stop_count,

        "tp_full_count": int(close_reason_counts.get("TAKE_PROFIT_FULL", 0)),
        "stop_loss_count": int(close_reason_counts.get("STOP_LOSS", 0)),
        "forced_close_count": int(close_reason_counts.get("forced_close_end", 0)),

        "score": score,
    }


# ============================================================
# Resume support
# ============================================================

if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done_keys = set(previous["key"].astype(str))
    results = previous.to_dict("records")
    print(f"Resuming | completed={len(done_keys):,}")
else:
    done_keys = set()
    results = []


# ============================================================
# Grid
# ============================================================

grid = list(itertools.product(
    STOP_LOSS_PCTS,
    EXIT_PLANS,
    ALLOW_RULE_CLOSE_VALUES,
))

expected = (
    len(STOP_LOSS_PCTS)
    * len(EXIT_PLANS)
    * len(ALLOW_RULE_CLOSE_VALUES)
)

print(f"Expected combinations: {expected:,}")
print(f"Total combinations: {len(grid):,}")
print(f"Already completed: {len(done_keys):,}")
print(f"Remaining: {len(grid) - len(done_keys):,}")

# ============================================================
# Runtime estimate helper
# ============================================================

def estimate_runtime(seconds_per_iter: float, remaining: int):
    total_sec = seconds_per_iter * remaining
    return total_sec / 3600.0


# ============================================================
# Run optimizer
# ============================================================

t0 = time.time()
new_runs = 0

for (
    stop_loss_pct,
    exit_plan,
    allow_rule_close,
) in tqdm(grid, desc="Optimizing fast exits"):

    key = (
        f"sl_{_pct_name(stop_loss_pct)}__"
        f"plan_{exit_plan['name']}__"
        f"ruleclose_{allow_rule_close}__"
        f"sizing_{SIZING_CONFIG['name']}"
    )

    if key in done_keys:
        continue

    iter_t0 = time.time()

    exit_cfg = make_exit_cfg(
        stop_loss_pct=stop_loss_pct,
        exit_plan=exit_plan,
        allow_rule_close=allow_rule_close,
    )

    cfg = make_sim_config(exit_cfg)

    res = run_simulation(
        df=merged,
        strategy=fast_strategy,
        cfg=cfg,
        time_col="t",
        price_col="close",
    )

    params = {
        "key": key,
        "stop_loss_pct": stop_loss_pct,
        "exit_plan": exit_plan["name"],
        "exit_category": exit_plan["category"],
        "allow_rule_close": allow_rule_close,
        "sizing_mode": SIZING_CONFIG["name"],
    }

    results.append(summarize_exit_result(res, params))
    done_keys.add(key)
    new_runs += 1

    if new_runs % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

        elapsed_min = (time.time() - t0) / 60
        avg_sec = (time.time() - t0) / max(new_runs, 1)
        remaining = len(grid) - len(done_keys)
        eta_hours = estimate_runtime(avg_sec, remaining)

        print(
            f"Checkpoint saved | new_runs={new_runs:,} | "
            f"total_done={len(done_keys):,} | elapsed={elapsed_min:.1f} min | "
            f"avg={avg_sec:.2f}s/run | ETA={eta_hours:.2f}h"
        )


pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print(f"Done. Saved: {RESULTS_PATH}")


# ============================================================
# View best results
# ============================================================

opt = pd.read_csv(RESULTS_PATH)

valid = opt[opt["num_trades"] >= MIN_TRADES_FOR_RANKING].copy()

best_by_profit = valid.sort_values(
    ["pnl", "final_equity", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_score = valid.sort_values(
    ["score", "pnl", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_winrate = valid.sort_values(
    ["win_rate", "pnl", "num_trades"],
    ascending=False,
)

best_by_profit_factor = valid.sort_values(
    ["profit_factor", "pnl", "num_trades"],
    ascending=False,
)

print("\nBEST BY PROFIT")
display(best_by_profit.head(30))

print("\nBEST BY RISK-ADJUSTED SCORE")
display(best_by_score.head(30))

print("\nBEST BY WIN RATE")
display(best_by_winrate.head(30))

print("\nBEST BY PROFIT FACTOR")
display(best_by_profit_factor.head(30))

print("\nSUMMARY BY EXIT CATEGORY")
display(
    valid.groupby("exit_category")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)

print("\nSUMMARY BY STOP LOSS")
display(
    valid.groupby("stop_loss_pct")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)

print("\nSUMMARY BY RULE CLOSE MODE")
display(
    valid.groupby("allow_rule_close")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)

In [ ]:
strategy = Strategy(
    open_rules_long=open_rules_long,
    close_rules_long=close_rules_long,
)

# res = run_simulation(
#     df=merged,
#     strategy=strategy,
#     cfg=SimConfig(
#         initial_cash=10_000,
#         max_open_trades=1,
#         fee_bps=10,
#         slippage_bps=1,
#         # sim_start=sim_start,
#         sim_start="2026-01-01",
#         sim_end="2026-05-18",
#         #sim_start="2026-05-10",
#         #sim_end="2026-05-15",
#         sim_tz=tz,
#     ),
#     time_col="t",
#     price_col="close",
# )

from simulation.simulator import (
    SimConfig,
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

cfg = SimConfig(
    initial_cash=10_000,
    max_open_trades=1,
    fee_bps=10,
    slippage_bps=1,

    sim_start="2026-01-01",
    sim_end="2026-05-18",
    sim_tz=tz,

    exit=TradeExitConfig(
        enabled=True,
        # Important:
        # if both TP and SL are touched in same candle, assume SL first
        intrabar_priority="stop_first",
        # Important:
        # disables ST2 close rule while TP/SL system is active
        allow_rule_close=False,
        
        # stop_loss=StopLossConfig(
        #     mode="entry_pct",
        #     value=1,   # 0.5% below entry for long
        # ),

        # sizing=PositionSizingConfig(
        #     mode="cash",  # use normal full cash sizing
        # ),

        # take_profits=(
        #     TakeProfitConfig(
        #         label="TP1",
        #         mode="entry_pct",
        #         value=2,       # +1.0%
        #         close_pct=100.0, # close full trade
        #     ),
        # ),
        stop_loss=StopLossConfig(
            mode="entry_pct",
            value=0.5,   # 0.5% below entry for long
        ),
        sizing=PositionSizingConfig(
            mode="cash",  # use normal full cash sizing
        ),
        take_profits=(
            TakeProfitConfig(
                label="TP1",
                mode="entry_pct",
                value=0.5,          # +0.5%
                close_pct=50.0,     # close 50% of remaining
                move_stop_mode="breakeven",
            ),

            TakeProfitConfig(
                label="TP2",
                mode="entry_pct",
                value=1,          # +1.0%
                close_pct=100.0,    # close rest
            ),
        ),
    ),
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=cfg,
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
# plot_indicators = make_plot_indicators(
#     base_specs=ind_1m,
#     mtf_specs={
#         "5m": ind_5m,
#         "15m": ind_15m,
#     },
#     mtf_include={
#         "5m": ["macd_8_21_5"],
#         "15m": ["macd_8_21_5"],
#     }
# )

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
}


plot_toggles = [
    # Optional EMA overlays
    PlotToggle("1m", "ema", visible="legendonly"),
    PlotToggle("5m", "ema", visible="legendonly"),
    PlotToggle("15m", "ema", visible="legendonly"),

    # Supertrend setting 1
    PlotToggle("1m", ST1_TAG, visible="legendonly"),
    PlotToggle("5m", ST1_TAG, visible="legendonly"),
    PlotToggle("15m", ST1_TAG, visible="legendonly"),

    # Supertrend setting 2
    PlotToggle("1m", ST2_TAG, visible="legendonly"),
    PlotToggle("5m", ST2_TAG, visible="legendonly"),
    PlotToggle("15m", ST2_TAG, visible="legendonly"),

    # Hide MACD
    PlotToggle("1m", "macd_8_21_5",  visible="legendonly"),
    PlotToggle("5m", "macd_8_21_5",  visible="legendonly"),
    PlotToggle("15m", "macd_8_21_5",  visible="legendonly"),

    # Hide RSI
    PlotToggle("1m", "rsi14", visible=False),
    PlotToggle("5m", "rsi14", visible=False),
    PlotToggle("15m", "rsi14", visible=False),
]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)


In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-05-06",
        # date_to="2026-05-10",
        date_from="2026-1-05",
        date_to="2026-1-16",
        height=1400,
        width=1900,
    ),
)

# plot_simulation(
#     df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
#     symbol=symbol,
#     interval="5m",
#     market=market,
#     trades=res.trades,
#     ma_windows=[20],
#     indicators=ind_5m,
#     plot_cfg=PlotConfig(
#         tz="Asia/Karachi",
#         date_from="2026-02-01",
#         date_to="2026-02-05",
#         height=1400,
#         width=1900
#     ),
# )